# EMA + RSI

## Contents

- [Configuration](#configuration)
  - [Setup](#setup)
  - [Automatic](#automatic)
  - [Manual](#manual)
  - [Final configuration](#final-configuration)
- [EMA + RSI](#ema-rsi-section)
  - [Backtesting](#backtesting)
  - [Grid search](#grid-search)
  - [Walk-forward analysis](#walk-forward-analysis)
  - [Monte Carlo simulations](#monte-carlo-simulations)
  - [Live signals](#live-signals)
- [Inverse EMA + RSI](#inverse-ema--rsi)
  - [Backtesting](#inv-backtesting)
  - [Grid search](#inv-grid-search)
  - [Walk-forward analysis](#inv-walk-forward)
  - [Monte Carlo simulations](#inv-monte-carlo)
  - [Live signals](#inv-live-signals)
- [Adaptive EMA + RSI](#adaptive-ema--rsi)
  - [Backtesting](#adp-backtesting)
  - [Grid search](#adp-grid-search)
  - [Walk-forward analysis](#adp-walk-forward)
  - [Monte Carlo simulations](#adp-monte-carlo)
  - [Live signals](#adp-live-signals)

EMA Crossover + RSI Filter \
A momentum strategy that trades in the direction of the cross: fast EMA(9) over slow EMA(21) \
It uses ATR(14) for a dynamic volatility-adaptive trailing stop. \
Filters false signals with RSI(14) to cut whipsaws in ranging markets. \
Works across timeframes; well suited to scalping and intraday.

How EMA+RSI Algorithm Determines Entry/Exit:
- Fast EMA (9) / Slow EMA (21) – standard for crypto.
- Long Entry: Fast EMA crosses above Slow EMA AND RSI(14) < 70 (not overbought).
- Short Entry: Fast EMA crosses below Slow EMA AND RSI(14) > 30 (not oversold).
- Exit: Reverse crossover (signal flip) OR price hits the ATR-based trailing stop.
- The RSI filter reduces whipsaws in ranging markets.

The RSI filter on ema / ema_inv is an optional, configurable filter:
- filter off entirely: EmaParams(rsi_filter=False)
- custom bounds: EmaParams(rsi_bullish=65.0, rsi_bearish=35.0)

## Configuration

### Setup

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
import dataclasses

from engine.backtester import Backtester
from engine.data_configurator import ACTIVE, load_data, save_result, LIVE_DIR
from engine.strategy_configurator import params_for, EXIT_PRESETS
from engine.trade_configurator import ACTIVE_TRADE
from engine.visualization import build_chart
from engine.evaluation import walk_forward, monte_carlo, grid_search, oracle_ceiling
from engine.live import LiveEngine

import pandas as pd
import plotly.express as px

### Automatic

In [ ]:
# Automatic config: project-wide defaults defined by the three configurators.
# Override any of them in the manual chapter below; leave its dicts empty to stay automatic.
DATA_CONFIG     = ACTIVE              # engine/data_configurator.py     (DataSpec)
STRATEGY_CONFIG = params_for("ema")   # engine/strategy_configurator.py (EmaParams — this notebook's family)
TRADING_CONFIG  = ACTIVE_TRADE        # engine/trade_configurator.py    (TradingConfig)
EXIT_POLICY     = None                # None -> each strategy's assigned default (exit_policy_for)

### Manual


*_CONFIG = Automatic defaults, with any Manual overrides layered on top:
- Leave *_OVERRIDES empty → *_CONFIG is pure Automatic.
- Fill it → Automatic baseline + your Manual overrides.

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}          # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
# manual override — STRATEGY signal knobs. Empty = automatic EmaParams().
STRATEGY_OVERRIDES = {}      # e.g. {"ema_fast": 12, "ema_slow": 26, "rsi_filter": False}
STRATEGY_CONFIG = dataclasses.replace(STRATEGY_CONFIG, **STRATEGY_OVERRIDES)

In [ ]:
# manual override — EXIT policy. None = each strategy's assigned default.
# Preset: EXIT_POLICY = EXIT_PRESETS["fixed_2pct_rr3"]()
# Custom: from engine.exits import CompositeExit, AtrStop, RrTarget
#         EXIT_POLICY = CompositeExit(AtrStop(1.5), RrTarget(2.0))
EXIT_POLICY = None

In [ ]:
# manual override — TRADE config (costs / sizing / leverage / direction). Empty = ACTIVE_TRADE.
# For direction: from engine.trade_configurator import TradeDirection
TRADE_OVERRIDES = {}         # e.g. {"leverage": 2.0, "direction": TradeDirection.LONG}
TRADING_CONFIG = dataclasses.replace(TRADING_CONFIG, **TRADE_OVERRIDES)

### Final configuration

In [ ]:
# Prepare the final inputs the rest of the notebook uses.
# Runs after overrides.
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval

In [ ]:
# Report exactly what data + which configs are in force downstream (manual or automatic).
_window = (f"{DATA_CONFIG.start} → {DATA_CONFIG.end or 'now'}"
           if DATA_CONFIG.is_range else f"last {DATA_CONFIG.num_candles}")
tc = TRADING_CONFIG
_exits = (", ".join(f"{k!r}: {v!r}" for k, v in STRATEGY_CONFIG.EXITS.items())
          if EXIT_POLICY is None else f"override → {EXIT_POLICY}")
print(f"Loaded {len(df):,} candles | {SYMBOL} {INTERVAL}m {DATA_CONFIG.category} | "
      f"{_window} | {df.index[0]:%Y-%m-%d %H:%M} → {df.index[-1]:%Y-%m-%d %H:%M} UTC")
print(f"Trade: initial_equity={tc.initial_equity}, position_size_bps={tc.position_size_bps}, "
      f"leverage={tc.leverage}, sizing_mode={tc.sizing_mode.value!r}, "
      f"risk_per_trade_bps={tc.risk_per_trade_bps}, direction={tc.direction.value!r}")
print("Strategy Parameters: "
      + ", ".join(f"{k}={v}" for k, v in dataclasses.asdict(STRATEGY_CONFIG).items()))
print(f"Strategy exits: {_exits}")

<a id="ema-rsi-section"></a>
## EMA + RSI

### Backtesting

In [ ]:
# Import EMA + RSI strategy
from engine.strategies import EMACrossoverStrategy
STRATEGY = EMACrossoverStrategy

In [ ]:
# Backtest EMA + RSI strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Per-trade dollar P&L
trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "avg_duration_min": round(t.duration.total_seconds() / 60, 1) if t.duration else None,
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])
display(trades_pnl.head())

# Save metrics (JSON) + per-trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG, name="ema_rsi")

In [ ]:
# Theoretical ceiling
# For OHLCV-only crypto: ~5–15% is a realistic edge; under ~5% is noise.
_ceil_bps, _chain = oracle_ceiling(df, cost_bps=TRADING_CONFIG.total_cost_bps())
_skill = result.total_pnl_bps / _ceil_bps * 100 if _ceil_bps else 0.0
print(f"Theoretical ceiling : {_ceil_bps:+,.0f} bps  (full look-ahead, {TRADING_CONFIG.total_cost_bps():.0f} bps cost, {max(0, len(_chain) - 1)} trades)")
print(f"Strategy P&L   : {result.total_pnl_bps:+,.0f} bps")
print(f"Skill ratio    : {_skill:.1f}%  captured of what was theoretically possible")

In [ ]:
# EMA + RSI strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

### Grid search

In [ ]:
# Full grid search — Cartesian product across any of the four dimensions.
# Each grid is optional: uncomment the ones you want to sweep, leave the rest commented to hold them fixed.
# Row count = |strategy| × |trade| × |exit| × |data|, so keep grids tight.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.

gs = grid_search(
    STRATEGY,
    strategy_grid={"ema_fast": [19, 21, 56], "ema_slow": [22, 30, 76]},
    trade_grid={"leverage": [1.0, 2.0]},
    exit_grid=[None, "fixed_2pct_rr3", "chandelier_2atr"],
    data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

In [ ]:
# HEATMAP_METRIC is a configurable knob:
# flip between Sharpe / P&L / profit_factor / any grid_search column without editing the plot.
# Best value per ema_fast × ema_slow cell, across any other swept dimension.
# Renders only when the strategy grid is being swept.

HEATMAP_METRIC = "total_pnl_bps"   # any grid_search column
if {"ema_fast", "ema_slow"}.issubset(gs.columns):
    # Diverging colour split at the metric's breakeven: P&L/Sharpe at 0, profit_factor at 1, win_rate at 0.5.
    midpoint = {"profit_factor": 1.0, "win_rate": 0.5}.get(HEATMAP_METRIC, 0.0)
    px.imshow(
        gs.pivot_table(index="ema_fast", columns="ema_slow", values=HEATMAP_METRIC, aggfunc="max"),
        color_continuous_scale="RdYlGn", color_continuous_midpoint=midpoint, aspect="auto",
        labels=dict(x="ema_slow", y="ema_fast", color=HEATMAP_METRIC),
        title=f"In-sample {HEATMAP_METRIC} — {strategy.name} | {SYMBOL} {INTERVAL}m",
    ).show()
else:
    print("Heatmap needs ema_fast × ema_slow swept in the grid above — nothing to plot.")

### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the strategy grid every train window.
# TRAIN_BARS is an in-sample window swept for the best params.
# TEST_BARS is an out-of-sample window the winner is then tested on.
# OBJECTIVE  can be any sweep metric: total_pnl_bps | sharpe_approx | profit_factor | ...
# MIN_TRADES lets ignore in-sample combos with fewer trades (noise, not signal)

GRID = {"ema_fast": [5, 9, 13, 17], "ema_slow": [20, 30, 40, 50]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample (is) and their out-of-sample (oos) performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold (top) + cross-fold stability summary (bottom).
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window.
# nunique==1 => the optimiser locked the same value every fold; wide min..max / large std => jumpy.
display(wf.param_stability())
wf.param_stability_summary()

In [ ]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.

eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
(px.line(eq, labels={"value": "equity", "index": ""}, color_discrete_sequence=["steelblue"],
         title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m")
 .update_layout(showlegend=False)
 .update_traces(hovertemplate="%{x}<br>%{y:,.2f}<extra></extra>",
                hoverlabel=dict(bgcolor="steelblue", font_color="white"))
 .show())

In [ ]:
# Walk-forward OOS trades (entries/exits), drawn with the EMAs each fold actually traded.
# Entries sit on real crosses: each fold's winning EMAs are recomputed and shown only over that fold's test window.
# The lines step at fold boundaries (the visible jump) is the re-optimisation.
# See wf.folds_frame() for the per-window parameters.

prepared_wf = df.copy()
prepared_wf["ema_fast"] = float("nan")
prepared_wf["ema_slow"] = float("nan")
for f in wf.folds:
    prep = STRATEGY(dataclasses.replace(STRATEGY_CONFIG, **f.best_params)).prepare(df)
    seg = (df.index >= f.test_start) & (df.index <= f.test_end)
    prepared_wf.loc[seg, ["ema_fast", "ema_slow"]] = prep.loc[seg, ["ema_fast", "ema_slow"]]

build_chart(prepared_wf, trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | per-fold EMAs").show()

### Monte Carlo simulations

In [ ]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.

px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"}, color_discrete_sequence=["steelblue"],
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()

# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"}, color_discrete_sequence=["steelblue"],
             title="OOS max-drawdown distribution").show()

### Live signals

Live mode:
- runs the same strategy / config / costs as the backtest above
- generates signals: tells you when to enter / exit
- does not place orders
- the chart auto-refreshes every poll_seconds

Signal notification + sound:
- alerts you on each new entry/exit
- the first poll primes silently; alerts start from the next new signal
- browser = a banner + beep right in this cell's output (Safari/Chrome) \
  Runs via the notebook cell, not via CLI.
- desktop = a native macOS notification
- telegram reaches your phone by setting TELEGRAM_BOT_TOKEN / TELEGRAM_CHAT_ID in the environment

From the CLI:
- a terminal running the same loop
- prints a file link to the auto-refreshing chart
- add --notify to alert on each new signal: \
  python -m engine --strategy ema --mode live --interval 15 --poll 30 --notify desktop,telegram

From a notebook cell:
- prints a clickable chart link
- run() blocks the execution of the rest of the notebook until stopped

In [ ]:
live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
    notifiers="browser,desktop",        # ← edit here: browser / desktop / telegram, or None to disable
)
engine.run()    # to stop: interrupt the kernel/ use the Stop button

## Inverse EMA + RSI

Inverse EMA Crossover + RSI Filter \
Mean-reversion counterpart to EMA+RSI: fades the cross instead of riding it. \
Uses ATR(14) for dynamic trailing stops (adapts to volatility) and RSI(14) filter. \
Bets that EMA crosses mark momentum exhaustion, not continuation.

How Inverse EMA+RSI Algorithm Determines Entry/Exit:
- Fast EMA (9) / Slow EMA (21) — standard for crypto.
- Short Entry: Fast EMA crosses above Slow EMA AND RSI(14) > 30 (not oversold) — fading the bullish cross.
- Long Entry:  Fast EMA crosses below Slow EMA AND RSI(14) < 70 (not overbought) — fading the bearish cross.
- Exit: Opposite crossover (signal flip) OR price hits ATR-based trailing stop.
- Works best in range-bound / mean-reverting regimes; likely underperforms in strong trends.


<a id="inv-backtesting"></a>
### Backtesting

In [ ]:
# Import Inverse EMA + RSI strategy
from engine.strategies import InverseEMACrossoverStrategy
STRATEGY = InverseEMACrossoverStrategy

In [ ]:
# Backtest Inverse EMA + RSI strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Per-trade dollar P&L
trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "avg_duration_min": round(t.duration.total_seconds() / 60, 1) if t.duration else None,
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])
display(trades_pnl.head())

# Save metrics (JSON) + per-trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG, name="ema_rsi_inv")

In [ ]:
# Theoretical ceiling
# For OHLCV-only crypto: ~5–15% is a realistic edge; under ~5% is noise.
_ceil_bps, _chain = oracle_ceiling(df, cost_bps=TRADING_CONFIG.total_cost_bps())
_skill = result.total_pnl_bps / _ceil_bps * 100 if _ceil_bps else 0.0
print(f"Theoretical ceiling : {_ceil_bps:+,.0f} bps  (full look-ahead, {TRADING_CONFIG.total_cost_bps():.0f} bps cost, {max(0, len(_chain) - 1)} trades)")
print(f"Strategy P&L   : {result.total_pnl_bps:+,.0f} bps")
print(f"Skill ratio    : {_skill:.1f}%  captured of what was theoretically possible")

In [ ]:
# Inverse EMA + RSI strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

<a id="inv-grid-search"></a>
### Grid search

In [ ]:
# Full grid search — Cartesian product across any of the four dimensions.
# Each grid is optional: uncomment the ones you want to sweep, leave the rest commented to hold them fixed.
# Row count = |strategy| × |trade| × |exit| × |data|, so keep grids tight.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.

gs = grid_search(
    STRATEGY,
    strategy_grid={"ema_fast": [19, 21, 56], "ema_slow": [22, 30, 76]},
    trade_grid={"leverage": [1.0, 2.0]},
    exit_grid=[None, "fixed_2pct_rr3", "chandelier_2atr"],
    data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

In [ ]:
# HEATMAP_METRIC is a configurable knob:
# flip between Sharpe / P&L / profit_factor / any grid_search column without editing the plot.
# Best value per ema_fast × ema_slow cell, across any other swept dimension.
# Renders only when the strategy grid is being swept.

HEATMAP_METRIC = "total_pnl_bps"   # any grid_search column
if {"ema_fast", "ema_slow"}.issubset(gs.columns):
    # Diverging colour split at the metric's breakeven: P&L/Sharpe at 0, profit_factor at 1, win_rate at 0.5.
    midpoint = {"profit_factor": 1.0, "win_rate": 0.5}.get(HEATMAP_METRIC, 0.0)
    px.imshow(
        gs.pivot_table(index="ema_fast", columns="ema_slow", values=HEATMAP_METRIC, aggfunc="max"),
        color_continuous_scale="RdYlGn", color_continuous_midpoint=midpoint, aspect="auto",
        labels=dict(x="ema_slow", y="ema_fast", color=HEATMAP_METRIC),
        title=f"In-sample {HEATMAP_METRIC} — {strategy.name} | {SYMBOL} {INTERVAL}m",
    ).show()
else:
    print("Heatmap needs ema_fast × ema_slow swept in the grid above — nothing to plot.")

<a id="inv-walk-forward"></a>
### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the strategy grid every train window.
GRID = {"ema_fast": [5, 9, 13, 17], "ema_slow": [20, 30, 40, 50]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample (is) and their out-of-sample (oos) performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold (top) + cross-fold stability summary (bottom).
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window.
# nunique==1 => the optimiser locked the same value every fold; wide min..max / large std => jumpy.
display(wf.param_stability())
wf.param_stability_summary()

In [ ]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.

eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
(px.line(eq, labels={"value": "equity", "index": ""}, color_discrete_sequence=["steelblue"],
         title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m")
 .update_layout(showlegend=False)
 .update_traces(hovertemplate="%{x}<br>%{y:,.2f}<extra></extra>",
                hoverlabel=dict(bgcolor="steelblue", font_color="white"))
 .show())

In [ ]:
# Walk-forward OOS trades (entries/exits), drawn with the EMAs each fold actually traded.
# Entries sit on real crosses: each fold's winning EMAs are recomputed and shown only over that fold's test window.
# The lines step at fold boundaries (the visible jump) is the re-optimisation.
# See wf.folds_frame() for the per-window parameters.

prepared_wf = df.copy()
prepared_wf["ema_fast"] = float("nan")
prepared_wf["ema_slow"] = float("nan")
for f in wf.folds:
    prep = STRATEGY(dataclasses.replace(STRATEGY_CONFIG, **f.best_params)).prepare(df)
    seg = (df.index >= f.test_start) & (df.index <= f.test_end)
    prepared_wf.loc[seg, ["ema_fast", "ema_slow"]] = prep.loc[seg, ["ema_fast", "ema_slow"]]

build_chart(prepared_wf, trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | per-fold EMAs").show()

<a id="inv-monte-carlo"></a>
### Monte Carlo simulations

In [ ]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.

px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"}, color_discrete_sequence=["steelblue"],
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()

# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"}, color_discrete_sequence=["steelblue"],
             title="OOS max-drawdown distribution").show()

<a id="inv-live-signals"></a>
### Live signals

Live mode:
- runs the same strategy / config / costs as the backtest above
- generates signals: tells you when to enter / exit
- does not place orders
- the chart auto-refreshes every poll_seconds

Signal notification + sound:
- alerts you on each new entry/exit
- the first poll primes silently; alerts start from the next new signal
- browser = a banner + beep right in this cell's output (Safari/Chrome) \
  Runs via the notebook cell, not via CLI.
- desktop = a native macOS notification
- telegram reaches your phone by setting TELEGRAM_BOT_TOKEN / TELEGRAM_CHAT_ID in the environment

From the CLI:
- a terminal running the same loop
- prints a file link to the auto-refreshing chart
- add --notify to alert on each new signal: \
  python -m engine --strategy ema_inv --mode live --interval 15 --poll 30 --notify desktop,telegram

From a notebook cell:
- prints a clickable chart link
- run() blocks the execution of the rest of the notebook until stopped

In [ ]:
live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
    notifiers="browser,desktop",        # ← edit here: browser / desktop / telegram, or None to disable
)
engine.run()    # to stop: interrupt the kernel/ use the Stop button

<a id="adaptive-ema--rsi"></a>
## Adaptive EMA + RSI

RSI-driven regime switch — the EMA analogue of Adaptive SuperTrend's ADX switch. \
The EMA fast/slow cross is the raw signal; RSI decides whether to follow that cross or fade it:
- bullish cross + RSI not overbought (rsi < rsi_bullish) → follow → long
- bullish cross + RSI overbought (rsi ≥ rsi_bullish) → fade → short
- bearish cross + RSI not oversold (rsi > rsi_bearish) → follow → short
- bearish cross + RSI oversold (rsi ≤ rsi_bearish) → fade → long

So the same RSI bounds the plain ema strategy uses to skip an entry are used here to flip it: an overbought bullish cross becomes a mean-reversion short. Exits respect the regime captured at entry (follow vs fade), so a mid-trade RSI swing can't change a position's exit. \
Tune the switch from the manual chapter via the RSI bounds — e.g. STRATEGY_OVERRIDES = {"rsi_bullish": 65, "rsi_bearish": 35}. rsi_filter=False turns the switch off (plain follow-the-cross, same as base ema).

<a id="adp-backtesting"></a>
### Backtesting

In [ ]:
# Import Adaptive EMA + RSI strategy
from engine.strategies import AdaptiveEMACrossoverStrategy
STRATEGY = AdaptiveEMACrossoverStrategy

In [ ]:
# Backtest Adaptive EMA + RSI strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Per-trade dollar P&L
trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "avg_duration_min": round(t.duration.total_seconds() / 60, 1) if t.duration else None,
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])
display(trades_pnl.head())

# Save metrics (JSON) + per-trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG, name="ema_rsi_adaptive")

In [ ]:
# Theoretical ceiling
# For OHLCV-only crypto: ~5–15% is a realistic edge; under ~5% is noise.
_ceil_bps, _chain = oracle_ceiling(df, cost_bps=TRADING_CONFIG.total_cost_bps())
_skill = result.total_pnl_bps / _ceil_bps * 100 if _ceil_bps else 0.0
print(f"Theoretical ceiling : {_ceil_bps:+,.0f} bps  (full look-ahead, {TRADING_CONFIG.total_cost_bps():.0f} bps cost, {max(0, len(_chain) - 1)} trades)")
print(f"Strategy P&L   : {result.total_pnl_bps:+,.0f} bps")
print(f"Skill ratio    : {_skill:.1f}%  captured of what was theoretically possible")

In [ ]:
# Adaptive EMA + RSI strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

<a id="adp-grid-search"></a>
### Grid search

In [ ]:
# Full grid search — Cartesian product across any of the four dimensions.
# Each grid is optional: uncomment the ones you want to sweep, leave the rest commented to hold them fixed.
# Row count = |strategy| × |trade| × |exit| × |data|, so keep grids tight.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.

gs = grid_search(
    STRATEGY,
    strategy_grid={"ema_fast": [19, 21, 56], "ema_slow": [22, 30, 76]},
    trade_grid={"leverage": [1.0, 2.0]},
    exit_grid=[None, "fixed_2pct_rr3", "chandelier_2atr"],
    data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

In [ ]:
# HEATMAP_METRIC is a configurable knob:
# flip between Sharpe / P&L / profit_factor / any grid_search column without editing the plot.
# Best value per ema_fast × ema_slow cell, across any other swept dimension.
# Renders only when the strategy grid is being swept.

HEATMAP_METRIC = "total_pnl_bps"   # any grid_search column
if {"ema_fast", "ema_slow"}.issubset(gs.columns):
    # Diverging colour split at the metric's breakeven: P&L/Sharpe at 0, profit_factor at 1, win_rate at 0.5.
    midpoint = {"profit_factor": 1.0, "win_rate": 0.5}.get(HEATMAP_METRIC, 0.0)
    px.imshow(
        gs.pivot_table(index="ema_fast", columns="ema_slow", values=HEATMAP_METRIC, aggfunc="max"),
        color_continuous_scale="RdYlGn", color_continuous_midpoint=midpoint, aspect="auto",
        labels=dict(x="ema_slow", y="ema_fast", color=HEATMAP_METRIC),
        title=f"In-sample {HEATMAP_METRIC} — {strategy.name} | {SYMBOL} {INTERVAL}m",
    ).show()
else:
    print("Heatmap needs ema_fast × ema_slow swept in the grid above — nothing to plot.")

<a id="adp-walk-forward"></a>
### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the strategy grid every train window.
GRID = {"ema_fast": [5, 9, 13, 17], "ema_slow": [20, 30, 40, 50]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample (is) and their out-of-sample (oos) performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold (top) + cross-fold stability summary (bottom).
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window.
# nunique==1 => the optimiser locked the same value every fold; wide min..max / large std => jumpy.
display(wf.param_stability())
wf.param_stability_summary()

In [ ]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.

eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
(px.line(eq, labels={"value": "equity", "index": ""}, color_discrete_sequence=["steelblue"],
         title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m")
 .update_layout(showlegend=False)
 .update_traces(hovertemplate="%{x}<br>%{y:,.2f}<extra></extra>",
                hoverlabel=dict(bgcolor="steelblue", font_color="white"))
 .show())

In [ ]:
# Walk-forward OOS trades (entries/exits), drawn with the EMAs each fold actually traded.
# Entries sit on real crosses: each fold's winning EMAs are recomputed and shown only over that fold's test window.
# The lines step at fold boundaries (the visible jump) is the re-optimisation.
# See wf.folds_frame() for the per-window parameters.

prepared_wf = df.copy()
prepared_wf["ema_fast"] = float("nan")
prepared_wf["ema_slow"] = float("nan")
for f in wf.folds:
    prep = STRATEGY(dataclasses.replace(STRATEGY_CONFIG, **f.best_params)).prepare(df)
    seg = (df.index >= f.test_start) & (df.index <= f.test_end)
    prepared_wf.loc[seg, ["ema_fast", "ema_slow"]] = prep.loc[seg, ["ema_fast", "ema_slow"]]

build_chart(prepared_wf, trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | per-fold EMAs").show()

<a id="adp-monte-carlo"></a>
### Monte Carlo simulations

In [ ]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.

px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"}, color_discrete_sequence=["steelblue"],
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()

# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"}, color_discrete_sequence=["steelblue"],
             title="OOS max-drawdown distribution").show()

<a id="adp-live-signals"></a>
### Live signals

Live mode:
- runs the same strategy / config / costs as the backtest above
- generates signals: tells you when to enter / exit
- does not place orders
- the chart auto-refreshes every poll_seconds

Signal notification + sound:
- alerts you on each new entry/exit
- the first poll primes silently; alerts start from the next new signal
- browser = a banner + beep right in this cell's output (Safari/Chrome) \
  Runs via the notebook cell, not via CLI.
- desktop = a native macOS notification
- telegram reaches your phone by setting TELEGRAM_BOT_TOKEN / TELEGRAM_CHAT_ID in the environment

From the CLI:
- a terminal running the same loop
- prints a file link to the auto-refreshing chart
- add --notify to alert on each new signal: \
  python -m engine --strategy ema_adaptive --mode live --interval 15 --poll 30 --notify desktop,telegram

From a notebook cell:
- prints a clickable chart link
- run() blocks the execution of the rest of the notebook until stopped

In [ ]:
live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
    notifiers="browser,desktop",        # ← edit here: browser / desktop / telegram, or None to disable
)
engine.run()    # to stop: interrupt the kernel/ use the Stop button